In [ ]:
# ============================================================
# EfficientNetB0 ASL (29 classes) — 1 GPU, Float32 (NO FP16)
# Giữ cấu trúc như ResNet: safe_map, sanity batch, callbacks, plots, CM, save .keras
# ============================================================
import os, json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =========================
# GPU SETUP (1 GPU + memory growth)
# =========================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("Using single GPU:", gpus[0])
    except Exception as e:
        print("GPU setup warning:", e)
else:
    print("No GPU found — training on CPU")

# Dùng float32 để tránh bất ổn FP16 (NaN)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")

# -----------------------------
# CONFIG (SỬA 3 ĐƯỜNG DẪN NÀY)
# -----------------------------
TRAIN_DIR = "/kaggle/input/asl-alphabet/asl_split/train"   # <<< sửa
VAL_DIR   = "/kaggle/input/asl-alphabet/asl_split/val"     # <<< sửa
TEST_DIR  = "/kaggle/input/asl-alphabet/asl_split/test"    # <<< sửa

IMG_SIZE   = (224, 224)   # EfficientNetB0 default
BATCH_SIZE = 64
EPOCHS     = 50
SEED       = 123

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

# -----------------------------
# HÀM QUÉT FILES + LABELS
# -----------------------------
def list_image_files_with_labels(root_dir):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root_dir}")
    class_names = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if len(class_names) == 0:
        raise ValueError(f"No class subfolders found under: {root_dir}")
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    filepaths, labels = [], []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for c in class_names:
        for p in (root / c).rglob("*"):
            if p.suffix.lower() in exts:
                filepaths.append(str(p)); labels.append(class_to_idx[c])
    return filepaths, labels, class_names

# -----------------------------
# ĐỌC 3 BỘ DỮ LIỆU
# -----------------------------
train_files, train_labels, train_classes = list_image_files_with_labels(TRAIN_DIR)
val_files,   val_labels,   val_classes   = list_image_files_with_labels(VAL_DIR)
test_files,  test_labels,  test_classes  = list_image_files_with_labels(TEST_DIR)

if train_classes != val_classes or train_classes != test_classes:
    raise ValueError(
        "Class folders in TRAIN / VAL / TEST are not identical or not in the same order.\n"
        f"TRAIN: {train_classes}\nVAL  : {val_classes}\nTEST : {test_classes}\n"
        "Please ensure all three datasets contain the same 29 class subfolders with identical names."
    )

class_names = train_classes
num_classes = len(class_names)
print(f"[OK] Classes (29 expected): {num_classes}")
print(class_names)
print(f"Train images: {len(train_files)}")
print(f"Val   images: {len(val_files)}")
print(f"Test  images: {len(test_files)}")
assert num_classes == 29, "Expected 29 classes (26 letters + delete, nothing, space)."

In [ ]:
# -----------------------------
# TF.DATA PIPELINES (safe_map + deterministic)
# -----------------------------
AUTOTUNE = tf.data.AUTOTUNE

def decode_img_onehot(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32) / 255.0   # scale [0,1]
    y = tf.one_hot(tf.cast(label, tf.int32), depth=num_classes)
    return img, y

def safe_map(x, y):
    tf.debugging.assert_all_finite(x, "Found NaN/Inf in images")
    tf.debugging.assert_all_finite(y, "Found NaN/Inf in labels")
    return x, y

augment = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.08, fill_mode="reflect", seed=SEED),
    layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),
    layers.RandomTranslation(0.10, 0.10, fill_mode="reflect", seed=SEED),
], name="augment")

def make_ds(files, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(10000, len(files)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_img_onehot, num_parallel_calls=AUTOTUNE)
    ds = ds.map(safe_map, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.cache()
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files,   val_labels,   training=False)
test_ds  = make_ds(test_files,  test_labels,  training=False)

# -----------------------------
# METRIC: Macro-F1 (từ confusion matrix tích lũy)
# -----------------------------
class MacroF1(keras.metrics.Metric):
    def __init__(self, num_classes, name="f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.cm = self.add_weight(
            name="confusion_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_labels = tf.argmax(y_true, axis=1, output_type=tf.int32)
        y_pred_labels = tf.argmax(y_pred, axis=1, output_type=tf.int32)
        cm_batch = tf.math.confusion_matrix(
            y_true_labels, y_pred_labels, num_classes=self.num_classes, dtype=tf.float32
        )
        self.cm.assign_add(cm_batch)

    def result(self):
        tp = tf.linalg.diag_part(self.cm)
        fp = tf.reduce_sum(self.cm, axis=0) - tp
        fn = tf.reduce_sum(self.cm, axis=1) - tp
        precision = tf.math.divide_no_nan(tp, tp + fp)
        recall    = tf.math.divide_no_nan(tp, tp + fn)
        f1_per_c  = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)
        return tf.reduce_mean(f1_per_c)

    def reset_states(self):
        self.cm.assign(tf.zeros_like(self.cm))

# -----------------------------
# MODEL: EfficientNetB0 + Preprocess + GAP + Dropout + Dense(29)
# -----------------------------
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input

# Lưu ý: ta đang scale ảnh về [0,1]; preprocess_input của EfficientNetB0 mong đợi [0,255]
# nên nhân 255 trước khi gọi preprocess_input để khớp đúng thống kê ImageNet
inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = layers.Lambda(lambda z: preprocess_input(z * 255.0), name="effnet_preprocess")(inputs)

base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=x,
    pooling=None
)

# Ổn định BN trong backbone
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.momentum = 0.99
        layer.epsilon  = 1e-3

x = base.output
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="logits")(x)

model = keras.Model(inputs, outputs, name="efficientnetb0_asl")

optimizer = keras.optimizers.Adam(learning_rate=3e-4, epsilon=1e-7, clipnorm=1.0)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall"), MacroF1(num_classes)]
    # jit_compile=True  # bật khi đã ổn định
)

model.summary()

# -----------------------------
# CALLBACKS
# -----------------------------
ckpt_path = "checkpoints/efficientnetb0_asl_best.keras"
callbacks = [
    keras.callbacks.TerminateOnNaN(),
    keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=False, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, verbose=1
    ),
]

# -----------------------------
# SANITY CHECK 1 BATCH
# -----------------------------
bx, by = next(iter(train_ds))
sanity = model.evaluate(bx, by, verbose=0)
print("Sanity batch ->", dict(zip(model.metrics_names, sanity)))

# -----------------------------
# TRAIN
# -----------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# EVALUATE + CONFUSION MATRIX + CLASSIFICATION REPORT
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

print("\n=== TEST METRICS ===")
test_metrics = model.evaluate(test_ds, verbose=1)
for name, val in zip(model.metrics_names, test_metrics):
    print(f"{name}: {val:.6f}")

# ---- Plot curves ----
hist = history.history
def plot_curve(keys, title, fname):
    plt.figure(figsize=(7,5))
    for k in keys:
        if k in hist:
            plt.plot(hist[k], label=k)
    plt.title(title); plt.xlabel("Epoch"); plt.ylabel("Value")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(f"figures/{fname}", dpi=150, bbox_inches="tight"); plt.close()

plot_curve(["loss", "val_loss"], "Loss", "loss.png")
plot_curve(["accuracy", "val_accuracy"], "Accuracy", "accuracy.png")
plot_curve(["precision", "val_precision"], "Precision", "precision.png")
plot_curve(["recall", "val_recall"], "Recall", "recall.png")
plot_curve(["f1", "val_f1"], "Macro-F1", "f1.png")
print("✅ Saved training curves to figures/*.png")

# ---- Confusion Matrix ----
print("\n=== CONFUSION MATRIX & CLASSIFICATION REPORT ===")
y_true, y_pred = [], []
for batch_x, batch_y in test_ds:
    preds = model.predict(batch_x, verbose=0)
    y_pred.append(np.argmax(preds, axis=1))
    y_true.append(np.argmax(batch_y.numpy(), axis=1))
y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)

cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
cm_norm = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(10,8))
plt.imshow(cm_norm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix (Normalized)")
plt.colorbar(); plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.savefig("figures/confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.close()

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)
with open("figures/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("✅ Saved confusion matrix & report to figures/")

# ---- Save Model ----
model.save("artifacts/efficientnetb0_asl.keras")
with open("artifacts/label_map.json", "w", encoding="utf-8") as f:
    json.dump({i: c for i, c in enumerate(class_names)}, f, ensure_ascii=False, indent=2)
print("\n✅ Model saved -> artifacts/efficientnetb0_asl.keras")
print("✅ Label map saved -> artifacts/label_map.json")